## Environment Setup

In [ ]:
import sys
import logging
import deepxde as dde
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Force DeepXDE to use PyTorch
dde.backend.set_default_backend("pytorch")

# Configure notebook logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logging.info("Cell 1: PyTorch backend and logging initialized.")

## Spatio-temporal Domain

In [ ]:
def create_domain(num_domain=2000, num_boundary=500, num_initial=500):
    spatial_domain = dde.geometry.Rectangle([0.0, 0.0], [1.0, 1.0])
    temporal_domain = dde.geometry.TimeDomain(0.0, 1.0)
    geomtime = dde.geometry.GeometryXTime(spatial_domain, temporal_domain)

    def steel_interface(x, on_boundary):
        return on_boundary and np.isclose(x[1], 1.0)

    def electrolyte_bulk(x, on_boundary):
        return on_boundary and np.isclose(x[1], 0.0)

    def pipe_ends(x, on_boundary):
        return on_boundary and (np.isclose(x[0], 0.0) or np.isclose(x[0], 1.0))

    return {
        "geomtime": geomtime,
        "filters": {
            "steel": steel_interface,
            "bulk": electrolyte_bulk,
            "ends": pipe_ends
        },
        "sampling": {
            "domain": num_domain,
            "boundary": num_boundary,
            "initial": num_initial
        }
    }

domain_config = create_domain()
logging.info("Cell 2: Domain configuration and boundary filters mapped.")

## Dummy Data Generation 

In [ ]:
class API5LDataGenerator:
    def __init__(self, batch_size=1000, spatial_points=50):
        self.batch_size = batch_size
        self.spatial_points = spatial_points
        self.wt_min, self.wt_max = 0.25, 1.0   
        self.od_min, self.od_max = 4.0, 48.0   

    def generate_grf_profiles(self, length_scale=0.2):
        x = np.linspace(0, 1, self.spatial_points)
        X, Y = np.meshgrid(x, x)
        covariance_matrix = np.exp(-0.5 * ((X - Y) / length_scale)**2) + 1e-6 * np.eye(self.spatial_points)
        L = np.linalg.cholesky(covariance_matrix)
        standard_normals = np.random.randn(self.batch_size, self.spatial_points)
        normalized_profiles = 1 / (1 + np.exp(-(standard_normals @ L.T)))
        return torch.tensor(normalized_profiles, dtype=torch.float32)

    def generate_api5l_metadata(self):
        real_wt = torch.FloatTensor(self.batch_size).uniform_(self.wt_min, self.wt_max)
        real_od = torch.FloatTensor(self.batch_size).uniform_(self.od_min, self.od_max)
        
        # Simulated YOLO defect profile score (normalized area or severity score from 0 to 1)
        yolo_defect_score = torch.FloatTensor(self.batch_size).uniform_(0.0, 1.0)
        
        norm_wt = (real_wt - self.wt_min) / (self.wt_max - self.wt_min)
        norm_od = (real_od - self.od_min) / (self.od_max - self.od_min)
        
        # Stacking 3 features: [WT, OD, YOLO]
        meta_tensor = torch.stack((norm_wt, norm_od, yolo_defect_score), dim=1)
        return meta_tensor

generator = API5LDataGenerator(batch_size=1000, spatial_points=50)
soil_tensor = generator.generate_grf_profiles(length_scale=0.15)
fluid_tensor = generator.generate_grf_profiles(length_scale=0.3)
meta_tensor = generator.generate_api5l_metadata()

logging.info(f"Cell 3: Synthetic data generated. Meta tensor shape: {meta_tensor.shape}")

### Model Architecture

In [ ]:
class MIONet(nn.Module):
    def __init__(self, soil_dim, fluid_dim, meta_dim=3, trunk_dim=3, p_output=128):
        super(MIONet, self).__init__()
        try:
            self.branch_soil = nn.Sequential(
                nn.Linear(soil_dim, 128), nn.Tanh(),
                nn.Linear(128, 128), nn.Tanh(),
                nn.Linear(128, p_output)
            )
            self.branch_fluid = nn.Sequential(
                nn.Linear(fluid_dim, 128), nn.Tanh(),
                nn.Linear(128, 128), nn.Tanh(),
                nn.Linear(128, p_output)
            )
            self.branch_meta = nn.Sequential(
                nn.Linear(meta_dim, 64), nn.Tanh(),
                nn.Linear(64, p_output)
            )
            self.trunk = nn.Sequential(
                nn.Linear(trunk_dim, 128), nn.Tanh(),
                nn.Linear(128, 128), nn.Tanh(),
                nn.Linear(128, p_output)
            )
            
            # NEW: Map the combined latent space to 2 physical variables [C, phi]
            self.output_mapper = nn.Linear(p_output, 2)
            logging.info("MIONet updated for multi-variable output (C, phi).")
            
        except Exception as e:
            logging.error(f"Architecture initialization error: {e}")
            raise e

    def forward(self, inputs):
        try:
            x_soil, x_fluid, x_meta, x_trunk = inputs
            out_soil = self.branch_soil(x_soil)
            out_fluid = self.branch_fluid(x_fluid)
            out_meta = self.branch_meta(x_meta)
            out_trunk = self.trunk(x_trunk)
            
            # Element-wise merge of the branch parameters
            branch_combined = out_soil * out_fluid * out_meta
            
            # Combine the physical parameters with the space-time coordinates
            merged = branch_combined * out_trunk
            
            # Project to [Concentration, Potential]
            output = self.output_mapper(merged)
            return output
            
        except Exception as e:
            logging.error(f"Forward Pass Layer Error: {e}")
            raise e

# Instantiate model globally
model = MIONet(soil_dim=50, fluid_dim=50, meta_dim=3, trunk_dim=3)

### Residual Calculation

In [ ]:
def fick_diffusion_pde(x, u, D_global=1e-9):
    try:
        dC_dt = dde.grad.jacobian(u, x, i=0, j=2)
        d2C_dx2 = dde.grad.hessian(u, x, component=0, i=0, j=0)
        d2C_dy2 = dde.grad.hessian(u, x, component=0, i=1, j=1)
        return dC_dt - D_global * (d2C_dx2 + d2C_dy2)
    except Exception as e:
        logging.error(f"Fick PDE compilation exception: {e}")
        raise e

def butler_volmer_bc(x, u):
    try:
        C = u[:, 0:1]
        phi = u[:, 1:2]
        E_eq, alpha, F, R, T_surface, k_rate = -0.44, 0.5, 96485.0, 8.314, 298.15, 1e-5
        
        # Calculate raw physical overpotential
        overpotential = phi - E_eq
        
        # Numerical Stability Guard: Prevents torch.exp() from blowing up to infinity
        # during the first few random initialization epochs on the GPU
        overpotential_clamped = torch.clamp(overpotential, min=-5.0, max=1.5)
        
        exponent = (alpha * F) / (R * T_surface)
        reaction_rate = k_rate * C * torch.exp(exponent * overpotential_clamped)
        dC_dy = dde.grad.jacobian(u, x, i=0, j=1)
        return dC_dy + reaction_rate
    except Exception as e:
        logging.error(f"Butler-Volmer BC compilation exception: {e}")
        raise e

## Gradient Tracing and Optimization

In [ ]:
class PhysicsPathologyTracker:
    def __init__(self, check_every=500):
        self.check_every = check_every

    def audit_loss(self, step, pde_loss, bc_loss):
        if step % self.check_every == 0:
            if pde_loss < 1e-4 and bc_loss > 1e-2:
                logging.warning(
                    f"⚠️ SHORTCUT ALERT [Step {step}] | PDE Loss dropped to {pde_loss:.6f} "
                    f"but Chemistry BC remains high ({bc_loss:.6f}). The model is cheating."
                )

pathology_tracker = PhysicsPathologyTracker(check_every=100)

## Training Loop

In [ ]:
import time

def train_operator(model, domain_config, data_tensors, epochs=1000, device="cpu", log_every=1):
    """
    Orchestrates the backward optimization passes with per-epoch timing and high-frequency logging.
    """
    try:
        model = model.to(device)
        geomtime = domain_config["geomtime"]
        
        # Sample points and map to the active hardware cluster
        interior_coords = torch.tensor(geomtime.random_points(domain_config["sampling"]["domain"]), dtype=torch.float32, requires_grad=True).to(device)
        boundary_coords = torch.tensor(geomtime.random_boundary_points(domain_config["sampling"]["boundary"]), dtype=torch.float32, requires_grad=True).to(device)
        
        x_soil, x_fluid, x_meta = [t.to(device) for t in data_tensors]
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        logging.info(f"Training actively initialized on target device: {device}")

        for epoch in range(1, epochs + 1):
            epoch_start = time.time()
            optimizer.zero_grad()
            
            # 1. Evaluate interior PDE loss (Fick's Second Law)
            u_int = model((x_soil[0:1], x_fluid[0:1], x_meta[0:1], interior_coords))
            res_pde = fick_diffusion_pde(interior_coords, u_int)
            loss_pde = torch.mean(res_pde ** 2)
            
            # 2. Evaluate chemical boundary condition loss (Butler-Volmer)
            u_bc = model((x_soil[0:1], x_fluid[0:1], x_meta[0:1], boundary_coords))
            res_bc = butler_volmer_bc(boundary_coords, u_bc)
            loss_bc = torch.mean(res_bc ** 2)
            
            # 3. Aggregation & Backpropagation
            total_loss = loss_pde + loss_bc
            total_loss.backward()
            optimizer.step()
            
            epoch_time = time.time() - epoch_start
            
            # Detailed Verbosity Trigger
            if epoch % log_every == 0 or epoch == 1:
                logging.info(
                    f"Epoch {epoch:04d}/{epochs} | "
                    f"Total Loss: {total_loss.item():.6f} | "
                    f"PDE: {loss_pde.item():.6f} | "
                    f"BC: {loss_bc.item():.6f} | "
                    f"Step Time: {epoch_time:.4f}s"
                )
                pathology_tracker.audit_loss(epoch, loss_pde.item(), loss_bc.item())
                
        return model
    except Exception as e:
        logging.critical(f"Execution loop failed at epoch {epoch}: {e}")
        raise e

## Hardware Calls

In [ ]:
# Hardware Detection for Kubeflow
execution_device = "cuda" if torch.cuda.is_available() else "cpu"

# Start Run
trained_physics_model = train_operator(
    model=model,
    domain_config=domain_config,
    data_tensors=(soil_tensor, fluid_tensor, meta_tensor),
    epochs=1000,
    device=execution_device
)